In [3]:
import os
os.chdir(r'C:\Users\PC\financial-news-sentiment')
print(os.getcwd())

C:\Users\PC\financial-news-sentiment


In [1]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import re
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import joblib
import os

nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...


True

In [ ]:
# Cell 2 — Load data
df = pd.read_csv('data/processed/eda_reviewed.csv')
print(df.shape)
print(df['sentiment_label'].value_counts())

(4922, 6)
sentiment_label
neutral     2926
positive    1375
negative     621
Name: count, dtype: int64


In [7]:
# Cell 3 — Drop fragment titles (fewer than 4 words)
df['word_count'] = df['title'].apply(lambda x: len(str(x).split()))
df = df[df['word_count'] >= 4].drop(columns=['word_count'])
print(f"Rows after fragment drop: {df.shape[0]}")

Rows after fragment drop: 4918


In [8]:
# Cell 4 — Text cleaning function
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return ' '.join(tokens)

df['cleaned_title'] = df['title'].apply(clean_text)

In [9]:
# Cell 5 — Sanity check
print(df[['title', 'cleaned_title']].head(10))
print(f"\nAny nulls in cleaned_title: {df['cleaned_title'].isna().sum()}")
print(f"Any empty strings: {(df['cleaned_title'] == '').sum()}")

                                               title  \
0  According to Gran , the company has no plans t...   
1  Technopolis plans to develop in stages an area...   
2  The international electronic industry company ...   
3  With the new production plant the company woul...   
4  According to the company 's updated strategy f...   
5  FINANCING OF ASPOCOMP 'S GROWTH Aspocomp is ag...   
6  For the last quarter of 2010 , Componenta 's n...   
7  In the third quarter of 2010 , net sales incre...   
8  Operating profit rose to EUR 13.1 mn from EUR ...   
9  Operating profit totalled EUR 21.1 mn , up fro...   

                                       cleaned_title  
0  according to gran the company ha no plan to mo...  
1  technopolis plan to develop in stage an area o...  
2  the international electronic industry company ...  
3  with the new production plant the company woul...  
4  according to the company s updated strategy fo...  
5  financing of aspocomp s growth aspocomp is agg... 

In [10]:
# Cell 6 — Train/test split
X = df['cleaned_title']
y = df['sentiment_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train size: {X_train.shape[0]}")
print(f"Test size: {X_test.shape[0]}")
print(f"\nTrain label distribution:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"\nTest label distribution:\n{y_test.value_counts(normalize=True).round(3)}")

Train size: 3934
Test size: 984

Train label distribution:
sentiment_label
neutral     0.594
positive    0.280
negative    0.126
Name: proportion, dtype: float64

Test label distribution:
sentiment_label
neutral     0.595
positive    0.279
negative    0.126
Name: proportion, dtype: float64


In [11]:
# Cell 7 — TF-IDF vectorization
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"Train matrix shape: {X_train_tfidf.shape}")
print(f"Test matrix shape: {X_test_tfidf.shape}")

Train matrix shape: (3934, 5000)
Test matrix shape: (984, 5000)


In [12]:
# Cell 8 — Save outputs
os.makedirs('data/processed', exist_ok=True)
os.makedirs('models', exist_ok=True)

df[['title', 'cleaned_title', 'sentiment_label', 'source']].to_csv(
    'data/processed/cleaned_dataset.csv', index=False
)

joblib.dump(tfidf, 'models/tfidf_vectorizer.pkl')
joblib.dump((X_train_tfidf, X_test_tfidf, y_train, y_test), 'models/train_test_split.pkl')

print("All outputs saved.")

All outputs saved.
